Setting Up the folders

In [11]:
!pip install mediapipe opencv-python
import os
import numpy as np
import cv2

# Create storage
os.makedirs('data/password', exist_ok=True)
os.makedirs('data/random', exist_ok=True)
print("Folders created. Local setup complete.")

Folders created. Local setup complete.


No external JS Bridge needed. Local Webcam uses Inline Display!

In [ ]:
# Utilizing IPython Display for inline webcam feed.

Now we'll modify the recording and testing cells to use this new visualization function.

In [12]:
import cv2
import numpy as np
import mediapipe as mp
import time
import os
from IPython.display import display, Image, clear_output

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

def get_mp_lip_distance(landmarks, image_height, image_width):
    if not landmarks:
        return 0.0
    # Mediapipe points 13 and 14 correspond to upper and lower inner lip centers
    top_lip = landmarks.landmark[13]
    bottom_lip = landmarks.landmark[14]
    dist = abs(top_lip.y * image_height - bottom_lip.y * image_height)
    return dist

def record_sample_interactive(label, sample_num):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open local webcam.")
        return
        
    print(f"Stand by... Recording {label} sample #{sample_num}. Look at the camera...")
    time.sleep(2)

    frames_data = []
    for i in range(60):
        ret, img = cap.read()
        if not ret: break
            
        # Convert the BGR image to RGB before processing.
        results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
        lip_dist = 0.0
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                lip_dist = get_mp_lip_distance(face_landmarks, img.shape[0], img.shape[1])
                # Draw face mesh (lips emphasis)
                mp_drawing.draw_landmarks(
                    image=img,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_LIPS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style())

        frames_data.append([lip_dist])
        
        # The Inline Bridge: Render to Notebook Output
        _, enc = cv2.imencode('.jpeg', img)
        display(Image(data=enc.tobytes()))
        clear_output(wait=True)

    cap.release()
    
    file_path = f'data/{label}/{sample_num}.npy'
    np.save(file_path, np.array(frames_data))
    print(f"Saved: {file_path}")

# --- EXECUTION ---
# Run this cell multiple times with label="password" and label="random" to collect initial data
record_sample_interactive(label="password", sample_num=1)
record_sample_interactive(label="random", sample_num=1)

AttributeError: module 'mediapipe' has no attribute 'solutions'

Training the model

In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models
import glob
import numpy as np

def get_dataset():
    X, y = [], []
    for f in glob.glob('data/password/*.npy'):
        X.append(np.load(f))
        y.append(1)
    for f in glob.glob('data/random/*.npy'):
        X.append(np.load(f))
        y.append(0)
    if not X:
        print("Warning: No data found.")
        return np.array([]), np.array([])
    return np.array(X), np.array(y)

X_train, y_train = get_dataset()

if len(X_train) == 0:
    print("Cannot train model: No dataset found. Record password/random samples first.")
else:
    model = models.Sequential([
        layers.Input(shape=(60, 1)),
        layers.LSTM(64),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=100, verbose=1)
    model.save('lip_model.h5')
    print("Model trained and saved!")

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6667 - loss: 0.6625
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.6667 - loss: 0.6389
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6667 - loss: 0.6260
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.6667 - loss: 0.6220
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6667 - loss: 0.6230
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6667 - loss: 0.6250
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6667 - loss: 0.6254
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6667 - loss: 0.6238
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6667 - loss: 0.6210
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.6667 - loss: 0.6177
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6667 - loss: 0.6150
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6667 - loss

Model trained and saved!


In [14]:
import cv2
import numpy as np
import os
import glob
import time
from IPython.display import display, Image, clear_output

def set_password():
    print("Stand by... Recording your password. Please perform your lip gesture now.")
    for f in glob.glob('data/password/*.npy'):
        os.remove(f)
        
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return
        
    time.sleep(2)
    frames_data = []
    
    for i in range(60):
        ret, img = cap.read()
        if not ret: break
            
        results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        lip_dist = 0.0
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                lip_dist = get_mp_lip_distance(face_landmarks, img.shape[0], img.shape[1])
                mp_drawing.draw_landmarks(img, face_landmarks, mp_face_mesh.FACEMESH_LIPS,
                                          None, mp_drawing_styles.get_default_face_mesh_contours_style())
            
        frames_data.append([lip_dist])
        
        # Notebook Bridge Rendering
        _, enc = cv2.imencode('.jpeg', img)
        display(Image(data=enc.tobytes()))
        clear_output(wait=True)
        
    cap.release()
    
    file_path = 'data/password/0.npy'
    np.save(file_path, np.array(frames_data))
    print(f"Password recorded and saved to: {file_path}")
    
    # Retrain Model Automatically
    X_train, y_train = get_dataset()
    if len(X_train) == 0:
        print("No data to train model.")
        return
    if X_train.ndim == 2: X_train = np.expand_dims(X_train, axis=-1)
    elif X_train.ndim == 1: X_train = np.expand_dims(X_train, axis=(0, -1))
        
    model = models.Sequential([
        layers.Input(shape=(60, 1)),
        layers.LSTM(64),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    print("Training model...")
    model.fit(X_train, y_train, epochs=100, verbose=0)
    model.save('lip_model.h5')
    print("Password set and model updated successfully!")

In [15]:
import cv2
import numpy as np
import os

def unlock():
    if not os.path.exists('lip_model.h5'):
        print("No trained model found. Please run set_password() first.")
        return
        
    print("Loading model for verification...")
    model = tf.keras.models.load_model('lip_model.h5')
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return
        
    buffer = []
    try:
        for i in range(150):  # Run verification for a fixed duration of frames (~5 seconds)
            ret, img = cap.read()
            if not ret: break
                
            results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            lip_dist = 0.0
            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    lip_dist = get_mp_lip_distance(face_landmarks, img.shape[0], img.shape[1])
                    mp_drawing.draw_landmarks(img, face_landmarks, mp_face_mesh.FACEMESH_LIPS,
                                              None, mp_drawing_styles.get_default_face_mesh_contours_style())
                    
            buffer.append([lip_dist])
            
            status_text = "Mouth your password..."
            color = (255, 255, 0)
            
            if len(buffer) > 60:
                buffer.pop(0)
                input_data = np.expand_dims(np.array(buffer), axis=0)
                score = model.predict(input_data, verbose=0)[0][0]
                
                if score > 0.90:
                    status_text = f"ACCESS GRANTED ({score:.2f})"
                    color = (0, 255, 0)
                else:
                    status_text = f"LOCKED ({score:.2f})"
                    color = (0, 0, 255)
                    
            cv2.putText(img, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            
            # Notebook Bridge Rendering
            _, enc = cv2.imencode('.jpeg', img)
            display(Image(data=enc.tobytes()))
            clear_output(wait=True)
            
            # Stop once access is granted to make it graceful
            if "GRANTED" in status_text:
                print("ACCESS GRANTED! Stopping stream.")
                time.sleep(2)
                break
                
    finally:
        cap.release()
        print("Session ended.")

# Start Unlock
unlock()

Loading model for verification...
Session ended.


NameError: name 'face_mesh' is not defined